# Fine-tune FunctionGemma for XKCD Search (Unsloth)

In [ ]:
%%capture
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers triton
!pip install datasets

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'google/functiongemma-270m-it',
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.9: Fast Gemma3 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


model.safetensors:   0%|          | 0.00/393M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/63.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/714 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Making `model.base_model.model.model` require gradients


In [ ]:
from datasets import load_dataset
import json, random

xkcd_dataset = load_dataset('olivierdehaene/xkcd', split='train')
print(f'Loaded {len(xkcd_dataset)} XKCD comics')

README.md: 0.00B [00:00, ?B/s]

dataset.jsonl:   0%|          | 0.00/12.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2630 [00:00<?, ? examples/s]

Loaded 2630 XKCD comics


In [ ]:
TOOLS = [{
    'type': 'function',
    'function': {
        'name': 'search_xkcd',
        'description': 'Search XKCD comics by topic',
        'parameters': {
            'type': 'object',
            'properties': {'query': {'type': 'string'}},
            'required': ['query']
        }
    }
}]

def create_training_data(dataset):
    data = []
    templates = ['Find xkcd about {}', 'Search for {} comics', 'Show me xkcd on {}']
    
    for item in dataset:
        title = item['title'].lower() if item.get('title') else 'unknown'
        query = random.choice(templates).format(title)
        
        messages = [
            {'role': 'user', 'content': query},
            {'role': 'assistant', 'content': None, 'tool_calls': [{
                'type': 'function',
                'function': {'name': 'search_xkcd', 'arguments': json.dumps({'query': title})}
            }]}
        ]
        
        text = tokenizer.apply_chat_template(messages, tools=TOOLS, tokenize=False, add_generation_prompt=False)
        data.append({'text': text})
    
    return data

train_data = create_training_data(xkcd_dataset)
print(f'Created {len(train_data)} examples')

Created 2630 examples


In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

dataset = Dataset.from_list(train_data)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = 'text',
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = 'adamw_8bit',
        output_dir = 'outputs',
    ),
)

trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2630 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,630 | Num Epochs = 1 | Total steps = 329
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 3,796,992 of 271,895,168 (1.40% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
10,6.558300
20,2.093900
30,0.771100
40,0.523600
50,0.474300
60,0.378800
70,0.284900
80,0.259900
90,0.292500
100,0.284800


wandb: WARNING URL not available in offline run


train/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
train/grad_norm,█▄▂▂▂▂▂▂▁▁▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,███▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁
train/loss,█▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,171786720010752.0
train/epoch,1
train/global_step,329
train/grad_norm,0.6261
train/learning_rate,1e-05
train/loss,0.281


TrainOutput(global_step=329, training_loss=0.5558240254236935, metrics={'train_runtime': 590.0554, 'train_samples_per_second': 4.457, 'train_steps_per_second': 0.558, 'total_flos': 171786720010752.0, 'train_loss': 0.5558240254236935, 'epoch': 1.0})

In [ ]:
model.save_pretrained('xkcd-functiongemma')
tokenizer.save_pretrained('xkcd-functiongemma')
print('Saved!')

model.push_to_hub('xkcd-functiongemma')
tokenizer.push_to_hub('xkcd-functiongemma')

Saved!


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 26.3kB / 15.2MB            

Saved model to https://huggingface.co/xkcd-functiongemma


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tiongemma/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...ctiongemma/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

In [ ]:
FastLanguageModel.for_inference(model)

messages = [{'role': 'user', 'content': 'Find xkcd about programming'}]
text = tokenizer.apply_chat_template(messages, tools=TOOLS, tokenize=False, add_generation_prompt=True)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors='pt').to('cuda'),
    max_new_tokens = 256,
    streamer = TextStreamer(tokenizer, skip_prompt=True),
)


<start_function_call>call:search_xkcd{                    {"query": "programming"}}<end_function_call><start_function_response>
